# Linear Models — M1 and M2 (Ridge)

Fits four Ridge regression variants and selects the best via rolling time-series CV.

| ID | Model | Features |
|----|-------|----------|
| M1-raw | Ridge — flags, raw target | `length_tier_ord` + complexity flags |
| M1-log | Ridge — flags, log target | Same features, target = `log(hours_creation)` |
| M2-raw | Ridge — video_type, raw target | `length_tier_ord` + `video_type` dummies |
| M2-log | Ridge — video_type, log target | Same features, target = `log(hours_creation)` |

Alpha tuned via `TimeSeriesSplit` within the training set (Layer 1 CV).
Winner selected by mean rolling CV RMSE across 3 folds (Layer 2 CV).
**Must beat B1 RMSE of 5.45h to be considered worthwhile.**

In [3]:
import duckdb
import numpy as np
import pandas as pd
import sys
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import TimeSeriesSplit

sys.path.append('..')
from src.constants import LENGTH_TIER_MAP, LENGTH_TIER_ORDER, LENGTH_TIER_ORDINAL

In [4]:
con = duckdb.connect('../data/workload.duckdb')
videos = con.execute('SELECT * FROM dim_videos_ml').df()
con.close()

videos_complete = videos[videos['is_complete'] == 1].reset_index(drop=True)
print(f"Total: {len(videos)} | Complete: {len(videos_complete)} | Excluded: {len(videos) - len(videos_complete)}")

Total: 126 | Complete: 98 | Excluded: 28


## Setup — Split and Feature Engineering

In [5]:
N_TEST = 25

MODEL_COLS = [
    'expected_length_mins',
    'complexity_new',
    'complexity_media_depth',
    'complexity_delivery_style',
]

df_split = (
    videos_complete[
        ['date_first', 'hours_creation', 'media_title', 'video_type', 'media_type']
        + MODEL_COLS
    ]
    .dropna(subset=['hours_creation'])
    .sort_values('date_first')
    .reset_index(drop=True)
)

n_total     = len(df_split)
cutoff_idx  = n_total - N_TEST
cutoff_date = df_split.loc[cutoff_idx, 'date_first']

train_df = df_split.iloc[:cutoff_idx].copy()
test_df  = df_split.iloc[cutoff_idx:].copy()

print(f"Train: {len(train_df)}  ({train_df['date_first'].min().date()} -> {train_df['date_first'].max().date()})")
print(f"Test:  {len(test_df)}   ({test_df['date_first'].min().date()} -> {test_df['date_first'].max().date()})")

Train: 73  (2023-02-13 -> 2025-01-20)
Test:  25   (2025-02-06 -> 2026-01-20)


In [6]:
def add_features(df):
    df = df.copy()
    df['length_tier']        = df['expected_length_mins'].astype('Int64').map(LENGTH_TIER_MAP)
    df['length_tier_ord']    = df['length_tier'].map(LENGTH_TIER_ORDINAL)
    df['log_hours_creation'] = np.log(df['hours_creation'])
    # video_type dummies for M2 — Review is the reference level (largest group, dropped)
    vtype_dummies = pd.get_dummies(df['video_type'], prefix='vtype').astype(int)
    vtype_dummies = vtype_dummies.drop(columns=['vtype_Review'], errors='ignore')
    df = pd.concat([df, vtype_dummies], axis=1)
    return df

train_df = add_features(train_df)
test_df  = add_features(test_df)

# Align dummies — test set may be missing a category seen in train so this creates the column in the case 
# where the test data doesnt
vtype_cols = [c for c in train_df.columns if c.startswith('vtype_')]
for col in vtype_cols:
    if col not in test_df.columns:
        test_df[col] = 0

print("Feature engineering complete.")
print(f"video_type dummies: {vtype_cols}")

Feature engineering complete.
video_type dummies: ['vtype_Playthrough', 'vtype_Rankings']


In [7]:
#pd.crosstab(train_df['length_tier'],train_df['length_tier_ord'])

In [8]:
#train_df[['media_title','video_type','hours_creation','log_hours_creation']]

## Feature Sets and Helpers

In [9]:
# M1 features: ordinal length tier + complexity flags
M1_FEATURES = [
    'length_tier_ord',
    'complexity_new',
    'complexity_media_depth',
    'complexity_delivery_style',
]

# M2 features: ordinal length tier + video_type dummies (Review dropped as reference)
M2_FEATURES = ['length_tier_ord'] + vtype_cols

# Alpha candidates for RidgeCV — log-spaced from very small to very large
#chose Ridge as default, Lasso zeros out coefficients and with a small feature set, this is not practical
#start at 10^-3 =.001 (almost plain OLS where coefficents totally driven by data)
#end at 10^4=10,000 (strong Ridge regularization, coefficeients almost all shrunk to 0)
#50 values in the range
ALPHAS = np.logspace(-3, 4, 50)


# Baseline to beat
B1_RMSE = 5.45

def rmse(actual, predicted):
    return np.sqrt(np.mean((predicted - actual) ** 2))

## Layer 2 — Rolling CV (Model Selection)

Three expanding folds over the training set. For each fold, alpha is tuned via `TimeSeriesSplit` on that fold's training window (Layer 1), then RMSE is computed on the fold's test window. The mean RMSE across folds determines the winning variant.

In [10]:
def rolling_cv_ridge(train_df, features, use_log, alphas=ALPHAS, n_splits=5):
    """
    Layer 2 rolling CV for one Ridge variant.

    For each fold:
      - Layer 1: RidgeCV tunes alpha on fold's training window using TimeSeriesSplit
      - Predicts on fold's test window
      - If use_log=True, fits on log(y) and back-transforms predictions via exp()
      - Returns per-fold RMSE in original hours scale
    """
    n        = len(train_df)
    fold_size = n // 4   # ~25% of training set per test window

    # Three expanding folds
    folds = [
        (int(n * 0.5), int(n * 0.5) + fold_size),
        (int(n * 0.5) + fold_size, int(n * 0.5) + 2 * fold_size),
        (int(n * 0.5) + 2 * fold_size, min(int(n * 0.5) + 3 * fold_size, n)),
    ]

    fold_rmses = []

    for fold_idx, (test_start, test_end) in enumerate(folds):
        fold_train = train_df.iloc[:test_start].dropna(subset=features + ['hours_creation'])
        fold_test  = train_df.iloc[test_start:test_end].dropna(subset=features + ['hours_creation'])

        if len(fold_train) < 10 or len(fold_test) == 0:
            continue

        X_tr = fold_train[features].astype(float).values
        X_te = fold_test[features].astype(float).values
        y_tr = np.log(fold_train['hours_creation'].values) if use_log else fold_train['hours_creation'].values
        y_te = fold_test['hours_creation'].values  # always original scale for RMSE

        # Layer 1: tune alpha within fold's training window
        tscv   = TimeSeriesSplit(n_splits=n_splits)
        model  = RidgeCV(alphas=alphas, cv=tscv)
        model.fit(X_tr, y_tr)

        preds = model.predict(X_te)
        if use_log:
            preds = np.exp(preds)

        fold_rmse = rmse(y_te, preds)
        fold_rmses.append(fold_rmse)
        print(f"  Fold {fold_idx+1}: train n={len(fold_train)}, test n={len(fold_test)}, alpha={model.alpha_:.4f}, RMSE={fold_rmse:.2f}h")

    mean_rmse = np.mean(fold_rmses)
    print(f"  Mean CV RMSE: {mean_rmse:.2f}h")
    return mean_rmse, fold_rmses


print("Rolling CV function defined.")

Rolling CV function defined.


In [11]:
cv_results = {}

print("── M1-raw (flags, raw target) ──────────────────")
cv_results['M1-raw'] = rolling_cv_ridge(train_df, M1_FEATURES, use_log=False)

print("\n── M1-log (flags, log target) ──────────────────")
cv_results['M1-log'] = rolling_cv_ridge(train_df, M1_FEATURES, use_log=True)

print("\n── M2-raw (video_type, raw target) ─────────────")
cv_results['M2-raw'] = rolling_cv_ridge(train_df, M2_FEATURES, use_log=False)

print("\n── M2-log (video_type, log target) ─────────────")
cv_results['M2-log'] = rolling_cv_ridge(train_df, M2_FEATURES, use_log=True)

print("\n" + "=" * 50)
print(f"  {'Variant':<12}  {'Mean CV RMSE':>14}  {'vs B1 (5.45h)':>14}")
print(f"  {'-'*12}  {'-'*14}  {'-'*14}")
for name, (mean_rmse, _) in cv_results.items():
    delta = mean_rmse - B1_RMSE
    flag  = ' BEATS B1' if mean_rmse < B1_RMSE else ''
    print(f"  {name:<12}  {mean_rmse:>13.2f}h  {delta:>+13.2f}h{flag}")
print("=" * 50)

── M1-raw (flags, raw target) ──────────────────
  Fold 1: train n=36, test n=18, alpha=7.1969, RMSE=7.29h
  Fold 2: train n=54, test n=18, alpha=3.7276, RMSE=10.93h
  Fold 3: train n=72, test n=1, alpha=2.6827, RMSE=2.03h
  Mean CV RMSE: 6.75h

── M1-log (flags, log target) ──────────────────
  Fold 1: train n=36, test n=18, alpha=13.8950, RMSE=8.58h
  Fold 2: train n=54, test n=18, alpha=3.7276, RMSE=11.47h
  Fold 3: train n=72, test n=1, alpha=3.7276, RMSE=4.11h
  Mean CV RMSE: 8.05h

── M2-raw (video_type, raw target) ─────────────
  Fold 1: train n=36, test n=18, alpha=13.8950, RMSE=8.36h
  Fold 2: train n=54, test n=18, alpha=2.6827, RMSE=11.44h
  Fold 3: train n=72, test n=1, alpha=0.0010, RMSE=2.20h
  Mean CV RMSE: 7.33h

── M2-log (video_type, log target) ─────────────
  Fold 1: train n=36, test n=18, alpha=26.8270, RMSE=9.66h
  Fold 2: train n=54, test n=18, alpha=3.7276, RMSE=12.17h
  Fold 3: train n=72, test n=1, alpha=0.0010, RMSE=4.96h
  Mean CV RMSE: 8.93h

  Variant    

## Final Evaluation — Holdout

Retrain the winning variant on the full training set (Layer 1 alpha tuning included), then evaluate once on the held-out 25 videos. This number is the reported model performance.

In [ ]:
# ── Set winner here after reviewing CV results above ─────────────────────────
WINNER_FEATURES = M1_FEATURES   # <-- swap to M2_FEATURES if M2 wins
WINNER_LOG      = False          # <-- set True if log variant wins
WINNER_NAME     = 'M1-raw'       # <-- label for output
# ─────────────────────────────────────────────────────────────────────────────

train_clean = train_df.dropna(subset=WINNER_FEATURES + ['hours_creation'])
test_clean  = test_df.dropna(subset=WINNER_FEATURES + ['hours_creation'])

X_train = train_clean[WINNER_FEATURES].astype(float).values
X_test  = test_clean[WINNER_FEATURES].astype(float).values
y_train = np.log(train_clean['hours_creation'].values) if WINNER_LOG else train_clean['hours_creation'].values
y_test  = test_clean['hours_creation'].values

tscv          = TimeSeriesSplit(n_splits=5)
final_model   = RidgeCV(alphas=ALPHAS, cv=tscv)
final_model.fit(X_train, y_train)

final_preds = final_model.predict(X_test)
if WINNER_LOG:
    final_preds = np.exp(final_preds)

final_rmse = rmse(y_test, final_preds)

print(f"Winner: {WINNER_NAME}")
print(f"Tuned alpha: {final_model.alpha_:.4f}")
print(f"Holdout RMSE: {final_rmse:.2f}h  (B1 baseline: {B1_RMSE:.2f}h)")
print(f"Lift over B1: {B1_RMSE - final_rmse:.2f}h")

print(f"\nCoefficients:")
for feat, coef in zip(WINNER_FEATURES, final_model.coef_):
    print(f"  {feat:<35} {coef:+.4f}")